# RAG Application with LlamaIndex

## Introduction

**LlamaIndex** is a data framework for your LLM applications. It is designed to easily connect custom data sources to your LLM. While LangChain is a general-purpose framework for building LLM applications (chains, agents, etc.), LlamaIndex optimizes for **indexing** and **retrieving** data.

### Key Differences form LangChain
- **Simpler Interface**: LlamaIndex often requires less code to get a RAG system up and running.
- **Data-Centric**: It focuses heavily on the "Index" part—structuring your data for optimal retrieval.

In this notebook, we will build a RAG application to chat with our PDF documents using LlamaIndex.

## Step 1: Install Dependencies

We install LlamaIndex and a small local helper package for environment-variable fallback. In Colab, the API key should come from Secrets.

In [ ]:
%%capture
!pip install -q llama-index python-dotenv
print('Done')


## Step 2: Setup Environment

LlamaIndex uses OpenAI by default for generation and embeddings in this bonus notebook. Keep the key in Colab Secrets as `OPENAI_API_KEY`; never paste it into the notebook.

In [ ]:
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, add a secret named {name} and enable Notebook access."
        )
    return value

os.environ['OPENAI_API_KEY'] = get_secret('OPENAI_API_KEY')
print('OPENAI_API_KEY loaded')


## Step 3: Load Data

LlamaIndex makes data loading very simple with `SimpleDirectoryReader`. It automatically figures out how to parse files in a directory (PDFs, text files, etc.).

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load data from the 'pdfs' directory
reader = SimpleDirectoryReader(input_dir="pdfs")
documents = reader.load_data()

print(f"✅ Loaded {len(documents)} document pages")

## Step 4: Create Index

Now we create a `VectorStoreIndex` from our documents. This effectively:
1. Chunks the documents.
2. Embeds the chunks using OpenAI embeddings.
3. Stores them in an in-memory vector store (by default).

> **Note**: Since this is in-memory, if you restart the kernel, the index is lost and needs to be recreated (incurring embedding costs again). We will see how to persist it later.

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index (this happens in one line!)
index = VectorStoreIndex.from_documents(documents)

print("✅ Index created")

## Step 5: Querying

To ask questions, we create a "Query Engine" from our index. This engine handles the retrieval and LLM generation loop for us.

In [ ]:
# Create a query engine
query_engine = index.as_query_engine()

# Ask a question
response = query_engine.query("What is Byte Pair Encoding?")

print("Response:")
print(response)

### Inspecting the Source

We can check which documents were used to generate the answer.

In [ ]:
for node in response.source_nodes:
    print("--------------------------------------------------")
    print(f"Score: {node.score}")
    print(f"Source File: {node.metadata.get('file_name')}")
    print(f"Content: {node.text[:200]}...")

## Conclusion

As you can see, LlamaIndex abstracts away a lot of the complexity (chunking, embedding, retrieval setup) that we explicitly handled in the LangChain version. This makes it a great choice for getting up and running quickly with standard RAG pipelines.

## Persisting to Disk

By default, LlamaIndex stores the data in-memory. This means if you restart the kernel, you lose the index and have to rebuild it (re-embedding everything, which costs money).

We can persist the index to disk to avoid this.

In [ ]:
# Persist the index to a directory
index.storage_context.persist(persist_dir="./vector_db_llama")

print("✅ Index persisted to ./vector_db_llama")

### Loading from Disk

To load the index back, we use `StorageContext` and `load_index_from_storage`.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# Rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="./vector_db_llama")

# Load index from the storage context
loaded_index = load_index_from_storage(storage_context)
print("✅ Index loaded from disk")

# Verify it works
query_engine_loaded = loaded_index.as_query_engine()
response = query_engine_loaded.query("What is Byte Pair Encoding?")
print(response)

## 🎓 Student Challenge

Now it's your turn! 

**Goal**: Create a new RAG system using your *own* PDF documents.

1. Create a new folder named `student_pdfs` in this directory.
2. Upload 1-2 PDF files into that folder (e.g., a paper, a resume, a manual).
3. Use `SimpleDirectoryReader` to load documents from `student_pdfs`.
4. Create a new `VectorStoreIndex` from these documents.
5. Create a query engine and ask a question about your documents.

Write your code in the cells below.

In [ ]:
# 1. Load your documents from 'student_pdfs'


In [ ]:
# 2. Create an index


In [ ]:
# 3. Query your index
